# 00 - Load the raw data into DuckDB

**Goal.** Read the two CSV files from `data/raw/` and store them as tables in `game.duckdb`,
so that the next notebooks (and the SQL file) query a database instead of CSV files.

**Run this notebook first, then close it** (Kernel > Restart, or close the tab): DuckDB allows only one
process to write to the file, and an open connection blocks the other notebooks.

In [1]:
import duckdb

# The path is relative to the notebooks/ folder. The file is created if it does not exist.
con = duckdb.connect("../game.duckdb")

## Table `cookie_cats` (A/B test)

One row per player: `userid`, `version` (gate_30 or gate_40), `sum_gamerounds` (rounds played during
the first week), `retention_1` and `retention_7` (booleans: did the player come back on day 1 / day 7).

In [2]:
# read_csv_auto guesses separator, header and column types from the file.
# CREATE OR REPLACE makes the notebook safe to run again.
con.sql("""
    CREATE OR REPLACE TABLE cookie_cats AS
    SELECT * FROM read_csv_auto('../data/raw/cookie_cats.csv')
""")

con.sql("DESCRIBE cookie_cats").show()          # column names and types
con.sql("SELECT * FROM cookie_cats LIMIT 5").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ userid         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ version        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sum_gamerounds │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ retention_1    │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ retention_7    │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌────────┬─────────┬────────────────┬─────────────┬─────────────┐
│ userid │ version │ sum_gamerounds │ retention_1 │ retention_7 │
│ int64  │ varchar │     int64      │   boolean   │   boolean   │
├────────┼─────────┼────────────────┼─────────────┼─────────────┤
│    

In [3]:
# Quality check: userid should be unique (one row per player).
con.sql("""
    SELECT COUNT(*)               AS row_count,
           COUNT(DISTINCT userid) AS unique_players
    FROM cookie_cats
""").show()

┌───────────┬────────────────┐
│ row_count │ unique_players │
│   int64   │     int64      │
├───────────┼────────────────┤
│     90189 │          90189 │
└───────────┴────────────────┘



## Table `gaming_behavior` (player behaviour)

One row per player: sessions per week, session length, level, purchases, engagement label, etc.
Used for the segmentation in notebook 03.

In [4]:
con.sql("""
    CREATE OR REPLACE TABLE gaming_behavior AS
    SELECT * FROM read_csv_auto('../data/raw/online_gaming_behavior_insights.csv')
""")

con.sql("DESCRIBE gaming_behavior").show()
con.sql("SELECT * FROM gaming_behavior LIMIT 5").show()

┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ PlayerID                  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Age                       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Gender                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Location                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ GameGenre                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PlayTimeHours             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ InGamePurchases           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GameDifficulty            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [5]:
# Quality check: PlayerID should be unique.
con.sql("""
    SELECT COUNT(*)                AS row_count,
           COUNT(DISTINCT PlayerID) AS unique_players
    FROM gaming_behavior
""").show()

┌───────────┬────────────────┐
│ row_count │ unique_players │
│   int64   │     int64      │
├───────────┼────────────────┤
│     40034 │          40034 │
└───────────┴────────────────┘



In [6]:
# Close the connection so that the other notebooks can open the file.
con.close()